# PTEN Hands-on Lab B: RAG on Snowflake, two ways

**Goal (30 to 60 min):** give the DSA agent drilling knowledge from documents. Build the retrieval
foundation once, then orchestrate it two ways:

- **Level 1, LangChain on Snowflake:** the open-source reference point. Your code, your chain, and
  TruLens for observability.
- **Level 2, fully managed:** the same search service as a Cortex Agent tool, with tracing and
  evaluation built in.

See `reference/2026-10-26-rag-architecture-mapping.html` for the full component mapping.

| Block | What you do | Min |
|---|---|---|
| B1 | What RAG is, and the mapping | 5 to 10 |
| B2 | Stage, parse, chunk, Cortex Search | 10 |
| B3 | Level 1: LangChain chain + TruLens evaluation | 10 to 15 |
| B4 | Level 2: Search as a DSA agent tool | 5 to 10 |
| B5 | Deep dive / your own documents | 0 to 15 |

**Before you start:** finish Lab A first (it creates the warehouse, database and agent). Then, in this
notebook's settings, turn on the external access integration **`HOL_PTEN_PYPI_EAI`** so B3 can
`pip install` LangChain and TruLens, and restart the notebook. After that, **Run all** works.

**Two ways through every step:** like Lab A, each step opens with **why it matters** and **two ways to
run it**: the Python/SQL cells, or a **Cortex Code** prompt. Here the contrast is the point: Level 1 is
code you own, Level 2 is the same result with most of that code gone.

> The documents in `lab-b-rag/docs/` are **lab-authored samples** (stuck pipe prevention, hole
> cleaning, rig state definitions). They are illustrative, not operational guidance. B5 shows how
> to swap in your own documents from S3.

## B1. RAG in one paragraph

The model does not know PTEN's procedures. Rather than retrain it, retrieval-augmented generation
**finds** the relevant passages at question time and **passes them in** with the question. The four
core parts are an app, a context repository (governed, refreshed, chunked), search, and LLM
inference. Enterprises add an embedding model, a pipeline that keeps the knowledge base fresh,
evaluation and monitoring, and governance so restricted content only reaches the right people.

**Key idea for today:** the index is shared, only orchestration changes.

## B2. The shared retrieval foundation  *(Run now, about 10 min)*

> **Why it matters:** Whatever orchestration you choose, the knowledge base is the same: parsed, chunked, governed and indexed once. That is the part worth getting right; the rest is wiring.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below as written.
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `search-optimization` skill
>   loads automatically, then paste:
>
> ```text
> Upload the PDFs in lab-b-rag/docs to stage HOL_PTEN_ML.DOCS.RAW_DOCS, parse them with AI_PARSE_DOCUMENT in LAYOUT mode into DOCS.PARSED_DOCS, split them into 1500-character chunks with 200 overlap in DOCS.DOC_CHUNKS, and create Cortex Search service DOCS.DRILLING_DOCS_SEARCH on CHUNK with attributes TITLE and FILE_NAME on warehouse HOL_PTEN_WH.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
# Session + helpers. Works in a Snowsight Workspace notebook (active session) or locally
# with a named connection (SNOWFLAKE_CONNECTION_NAME).
import os, re
from pathlib import Path
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    from snowflake.snowpark import Session
    session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "default")).create()


def q(sql):
    """Run SQL and return a pandas DataFrame."""
    if sql.lstrip().upper().startswith(("SHOW", "DESC")):
        # to_pandas() on some SHOW outputs (e.g. SHOW SERVICES) fails on a column-count
        # mismatch; reading the result back through RESULT_SCAN is reliable.
        session.sql(sql).collect()
        return session.sql("SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))").to_pandas()
    return session.sql(sql).to_pandas()

REPO = Path.cwd().parent if Path.cwd().name.startswith("lab-") else Path.cwd()

def run_sql_file(rel):
    """Run a repo SQL file. Splits on ';' at line ends, keeping $$ ... $$ bodies intact."""
    text = (REPO / rel).read_text()
    stmts, buf, in_body = [], [], False
    for line in text.splitlines():
        if line.strip().startswith("--") and not in_body:
            continue
        buf.append(line)
        if line.count("$$") % 2 == 1:
            in_body = not in_body
        # a trailing "-- comment" after the ';' still ends the statement
        if not in_body and re.sub(r"\s--.*$", "", line).rstrip().endswith(";"):
            buf[-1] = re.sub(r"\s--.*$", "", buf[-1])
            stmts.append("\n".join(buf).strip().rstrip(";")); buf = []
    last = None
    for s in stmts:
        if s:
            last = session.sql(s).collect()
    print(f"ran {len(stmts)} statements from {rel}")
    return last

def use_lab():
    session.sql("USE WAREHOUSE HOL_PTEN_WH").collect()
    session.sql("USE DATABASE HOL_PTEN_ML").collect()
    print(session.get_current_account(), session.get_current_role(), "-> HOL_PTEN_WH / HOL_PTEN_ML")

In [ ]:
use_lab()

In [ ]:
# Land the sample PDFs on the internal stage. In production this is an external stage over S3.
for pdf in sorted((REPO / "lab-b-rag" / "docs").glob("*.pdf")):
    session.file.put(str(pdf), "@HOL_PTEN_ML.DOCS.RAW_DOCS/samples", auto_compress=False, overwrite=True)
q("LIST @HOL_PTEN_ML.DOCS.RAW_DOCS")

> **Vendor agnostic vs Snowflake native: parsing and chunking**
>
> | | Vendor agnostic | Snowflake native |
> |---|---|---|
> | **How** | PyPDF/Unstructured to extract text, LangChain RecursiveCharacterTextSplitter, run in a Python job | AI_PARSE_DOCUMENT (layout + OCR) and SPLIT_TEXT_RECURSIVE_CHARACTER in SQL |
> | **Pros** | Huge ecosystem of loaders; fine control over every step | One SQL statement per step; scales with the warehouse; results are governed tables |
> | **Cons** | Another job to schedule and scale; OCR and table layout need extra libraries; outputs live outside governance | Less knob-twiddling than a custom parser; very unusual formats may still need a UDF |
>
> **When to pick which:** native for PDFs, scans and Office documents. Keep a LangChain loader in a Snowpark UDF for odd formats; the chunk table and everything downstream stay the same.

In [ ]:
# Parse, chunk and create the Cortex Search service: runs lab-b-rag/rag_foundation.sql statement by statement (open the file to read the SQL)
run_sql_file("lab-b-rag/rag_foundation.sql")

In [ ]:
q("SELECT title, chunk_index, LEFT(chunk, 160) AS preview FROM DOCS.DOC_CHUNKS ORDER BY title, chunk_index LIMIT 8")

> **Vendor agnostic vs Snowflake native: embeddings and the index**
>
> | | Vendor agnostic | Snowflake native |
> |---|---|---|
> | **How** | Embedding API + FAISS/Pinecone/pgvector; your own refresh job | Cortex Search service over the chunk table |
> | **Pros** | Choose any embedding model and vector DB | Hybrid vector + keyword search with reranking; embeddings and refresh managed (TARGET_LAG); RBAC on the service |
> | **Cons** | Two more systems to secure and pay for; embeddings drift out of sync with the source; no keyword search unless you add it | Embedding model choice limited to what Cortex Search supports |
>
> **When to pick which:** native unless you need a specific embedding model the service does not offer. Either way, keep chunks in a table.

**Go look:** Snowsight » **AI & ML » Cortex Search » DRILLING_DOCS_SEARCH**. Use the playground to
search "overpull on connections" and see the hybrid scores.

## B3. Level 1: LangChain on Snowflake  *(Run now, about 10 to 15 min)*

This is the version most teams have seen in tutorials. `langchain-snowflake` gives two building blocks:

- `SnowflakeCortexSearchRetriever`: the retriever, backed by the Cortex Search service from B2
- `ChatSnowflake`: the chat model, backed by Cortex LLMs

You compose them with LangChain Expression Language (LCEL). Requires PyPI access
(`HOL_PTEN_PYPI_EAI` on the notebook's external access settings).

> **Why it matters:** This is the pattern most RAG tutorials show. We build it fully, including the part tutorials often skip: with code you own, you add evaluation and tracing yourself.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below as written.
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `machine-learning` skill
>   loads automatically, then paste:
>
> ```text
> Build a LangChain RAG chain with langchain-snowflake: SnowflakeCortexSearchRetriever over HOL_PTEN_ML.DOCS.DRILLING_DOCS_SEARCH (k=4) and ChatSnowflake with claude-sonnet-4-5, then instrument it with TruLens TruChain as app pten_drilling_rag, run three drilling questions and compute answer relevance, context relevance and groundedness in Snowflake.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
%pip install --quiet -U langchain-core langchain-snowflake trulens-core trulens-providers-cortex trulens-connectors-snowflake trulens-apps-langchain

In [ ]:
from langchain_snowflake import SnowflakeCortexSearchRetriever, ChatSnowflake
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

retriever = SnowflakeCortexSearchRetriever(
    session=session, service_name="HOL_PTEN_ML.DOCS.DRILLING_DOCS_SEARCH", database="HOL_PTEN_ML", schema="DOCS",
    k=4, auto_format_for_rag=True, content_field="CHUNK", join_separator="\n\n",
    fallback_to_page_content=True)

llm = ChatSnowflake(session=session, model="claude-sonnet-4-5", temperature=0.1, max_tokens=800)

prompt = ChatPromptTemplate.from_template("""
You are a drilling engineering assistant. Answer only from the context.
If the context does not cover the question, say so.

Context:
{context}

Question: {question}
""")

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm
print(rag_chain.invoke("What are the early warning signs of stuck pipe, and what should the driller do first?").content)

### Evaluation: why TruLens is required here

Snowflake cannot see inside a chain that runs in **your** Python process. To get traces and quality
metrics, you instrument the app with TruLens. It registers the app as an **External Agent** object
and writes spans to Snowflake, so the results still show up in **AI & ML » More tools » TruLens**.
The metrics below are LLM-judged: answer relevance, context relevance and groundedness (is the answer
supported by the retrieved text).

In [ ]:
import time
import pandas as pd
from datetime import datetime
from trulens.apps.langchain import TruChain
from trulens.connectors.snowflake import SnowflakeConnector
from trulens.core.run import RunConfig

session.sql("USE SCHEMA HOL_PTEN_ML.DOCS").collect()
tru_app = TruChain(rag_chain, app_name="pten_drilling_rag", app_version="langchain_v1",
                   connector=SnowflakeConnector(snowpark_session=session), main_method_name="invoke")

questions = pd.DataFrame({"query": [
    "What overpull on connections indicates a developing restriction?",
    "How should flow rate and RPM be set in an 8-3/4 inch lateral?",
    "What is the difference between slips-to-slips and weight-to-weight connection time?",
]})
run = tru_app.add_run(run_config=RunConfig(
    run_name=f"lab_b_{datetime.now():%Y%m%d%H%M%S}", dataset_name="pten_drilling_questions",
    source_type="DATAFRAME", dataset_spec={"input": "query"}))
run.start(input_df=questions)
while run.get_status() != "INVOCATION_COMPLETED":
    time.sleep(3)
run.compute_metrics(["answer_relevance", "context_relevance", "groundedness"])
print("Open AI & ML » More tools » TruLens » PTEN_DRILLING_RAG to see the scores")

**Go look:** Snowsight » **AI & ML » More tools » TruLens » PTEN_DRILLING_RAG**. Open one record: you see the
retrieval span, the generation span and each metric with the judge's reasoning.

> **Vendor agnostic vs Snowflake native: orchestration and evaluation**
>
> | | Vendor agnostic | Snowflake native |
> |---|---|---|
> | **How** | LangChain/LangGraph chain anywhere (laptop, Lambda, EKS) + TruLens or another eval tool | Cortex Agent with Cortex Search as a tool |
> | **Pros** | Every step visible and customizable; portable across LLM vendors; huge community | Planning, citations, tracing and evaluations built in; runs under the user's role; combines documents with data tools in one answer |
> | **Cons** | You own prompts, retries, version pinning and tracing; credentials wherever the chain runs; observability exists only because you added it | Less control over the exact chain; Cortex models only |
>
> **When to pick which:** LangChain is the right teaching and prototyping tool and a fine choice when the app already lives in a Python service. For the DSA assistant, where documents must sit next to scores and sensor data, the managed agent removes most of the code you just wrote.

## B4. Level 2: the same index as a DSA agent tool  *(Run now, about 5 to 10 min)*

No new index, no new chunks. We re-specify the Lab A agent with one more tool, `DrillingDocs`,
pointing at the same Cortex Search service. Then we ask the question that needs **both** data and documents.

> **Why it matters:** Same index, no new code path: one more tool on the agent. The payoff is the question that needs both the model scores and the practice guide, answered in one response with citations.
>
> **Two ways to run this step**
> - **Option A, plain Python/SQL:** run the cells below (the first one runs `lab-b-rag/add_search_to_agent.sql`).
> - **Option B, Cortex Code (CoCo):** open CoCo (Snowsight Workspaces, Desktop or CLI), the `agent-studio` skill
>   loads automatically, then paste:
>
> ```text
> Add a Cortex Search tool DrillingDocs to agent HOL_PTEN_ML.AGENT.DSA_AGENT using service HOL_PTEN_ML.DOCS.DRILLING_DOCS_SEARCH with title column TITLE, id column RELATIVE_PATH and max 4 results, keeping the existing tools, then ask it which rigs showed stuck-pipe risk on October 12 and what our practice guide says to check.
> ```
>
> Both paths create the same object names, so you can mix them and the later cells still work.

In [ ]:
# Add DrillingDocs to the DSA agent: runs lab-b-rag/add_search_to_agent.sql statement by statement (open the file to read the SQL)
run_sql_file("lab-b-rag/add_search_to_agent.sql")

In [ ]:
import json

def ask_agent(question, agent="HOL_PTEN_ML.AGENT.DSA_AGENT"):
    payload = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": question}]}]})
    raw = session.sql("SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?, FALSE)", params=[agent, payload]).collect()[0][0]
    content = json.loads(raw).get("content", [])
    print("Tools used:", [c["tool_use"]["name"] for c in content if c.get("type") == "tool_use"], "\n")
    print("\n".join(c["text"] for c in content if c.get("type") == "text"))

ask_agent("Which rigs showed stuck-pipe risk on October 12, and what does our practice guide say to check?")

**What happened:** the agent planned two steps on its own. It called `DrillingAnalyst` for the
rigs and scores and `DrillingDocs` for the practice guide, then wrote one answer with both. Open the
trace in **AI & ML » Agents » DSA_AGENT** to see it. No TruLens was needed: tracing is native.

The SPCS app from Lab A now answers this question too, with no redeploy, because the app calls the
agent by name.

> **Exercise for later: Level 1 vs Level 2 side by side**
>
> Ask the three TruLens questions from B3 through `ask_agent` and compare answers, citations and latency with the LangChain chain. Add them to the agent's evaluation set from A4.

## B5. Deep dive and your own documents  *(Optional)*

> **Exercise for later: swap in PTEN's documents**
>
> Create an external stage over the S3 prefix holding your drilling papers and requirements (`CREATE STAGE ... URL='s3://...' STORAGE_INTEGRATION=...  DIRECTORY=(ENABLE=TRUE AUTO_REFRESH=TRUE)`), point `rag_foundation.sql` at it, and re-run B2. The search service, the LangChain chain and the agent do not change.

> **Exercise for later: chunking experiments**
>
> Rebuild `DOC_CHUNKS` with 800 and 2,500 character chunks, create one search service per size, and rerun the TruLens run against each (`app_version='chunk_800'`, ...). Compare context relevance in AI & ML » More tools » TruLens.

> **Exercise for later: keep it fresh automatically**
>
> Add a stream on the stage directory table and a task that parses and chunks only new files, so the search service refreshes within its TARGET_LAG when a document lands in S3.

> **Exercise for later: governance**
>
> Add a `DOC_CLASS` attribute (public / internal / restricted) to the chunks, and use a row access policy or a search filter so restricted procedures only reach the roles that should see them.